# 🚀 IRIS Remote Coding Companion on Google Colab

**Hardware Specs:** 2 vCPUs • **12.7 GB RAM** • 100+ GB SSD • **$0 Free (No Credit Card Required)**

This notebook runs the **IRIS Remote Execution Companion** inside Google Colab's cloud environment, providing ample RAM and compute for Google Antigravity, git repositories, compilers, and unit tests.

### Your Permanent Companion Endpoint:
👉 **`https://irisforgecode.dpdns.org`** *(Never changes!)*

### Instructions:
1. Click the **Play (▶)** button on the code cell below.
2. Once connected, open IRIS and connect via **Coworkers ➔ Remote Cloud**!
3. Keep this Colab tab running while working on tasks.

In [ ]:
#@title Launch IRIS Remote Companion { vertical-output: true }
# 1. Clean up any previous runs
!pkill -f uvicorn 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true

import os, sys, subprocess, time, urllib.request

# 2. Configuration
COMPANION_SECRET = "iris_colab_secret_2026"
CLOUDFLARE_TUNNEL_TOKEN = "eyJhIjoiYzE3ZWJiZTFiODBiYTlkNmM1ZTNmMjE4ZGIzYzNlZDAiLCJ0IjoiZWVhYTJhZDgtYzY2OC00OTVjLTg3ZjMtZDYzOTM1YTQzZTY5IiwicyI6Ik4yUmlOakl5Wm1FdFlXUXlNaTAwTVRJMkxUaGlaRGN0WlRVM01tTmlPREpqWmpjMyJ9"
os.environ["IRIS_COMPANION_SECRET"] = COMPANION_SECRET
os.environ["ENVIRONMENT"] = "production"

# 3. Clone or update IRIS companion repository
print("[*] Setting up IRIS companion repository...")
if not os.path.exists("/content/IRIS"):
    subprocess.run(["git", "clone", "https://github.com/KIRTAN0902/IRIS.git", "/content/IRIS"], check=True)
else:
    subprocess.run(["git", "-C", "/content/IRIS", "pull"], check=True)

os.chdir("/content/IRIS/remote_companion")

# 4. Install dependencies
print("[*] Installing companion dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# 5. Install Cloudflare Tunnel (cloudflared)
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("[*] Installing Cloudflare Tunnel...")
    subprocess.run(
        "wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared",
        shell=True,
        check=True
    )

# 6. Start Uvicorn Companion Server in background
print("[*] Starting IRIS FastAPI Companion on port 7860...")
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", "7860"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)

# 7. Verify local server health
try:
    with urllib.request.urlopen("http://127.0.0.1:7860/health", timeout=5) as r:
        print("[✓] Companion FastAPI server is HEALTHY on port 7860!")
except Exception as e:
    print("[!] Warning: Health check on port 7860:", e)

# 8. Connect to Cloudflare Tunnel (Token Mode)
if CLOUDFLARE_TUNNEL_TOKEN:
    print("\n" + "="*60)
    print("🎉 CONNECTING PERMANENT TUNNEL TO: https://irisforgecode.dpdns.org")
    print(f"👉 IRIS_COMPANION_SECRET: {COMPANION_SECRET}")
    print("="*60 + "\n")
    !/usr/local/bin/cloudflared tunnel run --token $CLOUDFLARE_TUNNEL_TOKEN
else:
    print("[*] Establishing ephemeral public HTTPS tunnel...")
    !/usr/local/bin/cloudflared tunnel --url http://127.0.0.1:7860
